In [1]:
# Core RAG Frameworks
!pip install langchain
!pip install langchain-openai
!pip install langchain-community
!pip install langchain-text-splitters
!pip install langchain-experimental
!pip install langgraph

# Vector Stores
!pip install chromadb
!pip install langchain-chroma

# Import the Python SDK for Colab secrets
from google.colab import userdata


In [2]:
import json
import os
import time
from operator import itemgetter  # Used in exercises/solutions for LCEL key extraction
# LangChain is a framework for building LLM applications
# Reference: https://python.langchain.com/docs/get_started/introduction
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # OpenAI integrations
from langchain_chroma import Chroma  # Vector database for similarity search
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Smart text chunking
from langchain_core.documents import Document  # Document abstraction
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # Prompt templates
from langchain_core.messages import HumanMessage, AIMessage  # Chat history message types
from langchain_core.output_parsers import StrOutputParser  # Parse LLM output
from langchain_core.runnables import RunnablePassthrough  # Pass data through pipeline> lambda x:x

In [3]:
# ============================================================================
# PART 1: Data Ingestion & Vector Store Setup
# ============================================================================
print("\n" + "="*80)
print("PART 1: Data Ingestion Pipeline")
print("="*80)

# Load tickets
with open('./sample_data/synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)
print(f"✓ Loaded {len(tickets)} support tickets")

# Convert to LangChain Document objects
# Documents are the core abstraction in LangChain - they combine content with metadata
# Reference: https://python.langchain.com/docs/modules/data_connection/document_loaders/
documents = []
for ticket in tickets:
    # Create rich document with all context
    # TIP: Structure your content logically - LLMs understand formatted text better
    content = f"""
Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Date: {ticket['created_date']} to {ticket['resolved_date']}

Problem Description:
{ticket['description']}

Resolution:
{ticket['resolution']}
    """.strip()

    # Create Document with metadata
    # Metadata is crucial for filtering, citation, and source tracking
    # Best practice: Include all information you might want to filter or display later
    doc = Document(
        page_content=content,  # The actual text content
        metadata={  # Structured data about the document
            'ticket_id': ticket['ticket_id'],
            'title': ticket['title'],
            'category': ticket['category'],
            'priority': ticket['priority'],
            'source': f"Ticket {ticket['ticket_id']}"
        }
    )
    documents.append(doc)

print(f"✓ Created {len(documents)} documents with metadata")


PART 1: Data Ingestion Pipeline
✓ Loaded 22 support tickets
✓ Created 22 documents with metadata


In [4]:
# Initialize OpenAI embeddings
# Embeddings convert text into vectors for semantic search
# Reference: https://platform.openai.com/docs/guides/embeddings

# Set your OpenAI API key from Colab secrets
# IMPORTANT: Add your OpenAI API key to Colab secrets under the name 'OPENAI_API_KEY'.
# Click on the 'key' icon in the left sidebar -> Secrets -> Add new secret
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_APIKEY')

print("\nInitializing OpenAI embedding model...")
embeddings = OpenAIEmbeddings(
    model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
)
print("✓ OpenAI embedding model ready")

# Build vector store using Chroma
# Chroma is an open-source vector database optimized for AI applications
# It stores embeddings and enables fast similarity search
# Reference: https://docs.trychroma.com/
print("\nBuilding Chroma vector store...")
# Always rebuild from scratch so repeated runs don't accumulate duplicate documents.
import shutil
persist_directory = "./rag_vectorstore"
if os.path.exists(persist_directory):
    try:
        shutil.rmtree(persist_directory)
    except PermissionError:
        # Windows may keep Chroma files locked briefly from a prior run.
        # Fall back to a unique directory so the demo can continue.
        persist_directory = f"./rag_vectorstore_{int(time.time())}"
        print(f"⚠ Vector store directory is locked; using {persist_directory} instead")
vector_store = Chroma.from_documents(
    documents=documents,  # Our support ticket documents
    embedding=embeddings,  # Embedding function to use
    collection_name="supportdesk_rag",  # Name for this collection
    persist_directory=persist_directory,  # Where to save the database
    collection_metadata={"hnsw:space": "cosine"}  # Use cosine distance for similarity search
)
print("✓ Vector store created and persisted")



Initializing OpenAI embedding model...
✓ OpenAI embedding model ready

Building Chroma vector store...
✓ Vector store created and persisted


In [5]:
# ============================================================================
# PART 2: Create Retriever
# ============================================================================
print("\n" + "="*80)
print("PART 2: Setting Up Retriever")
print("="*80)

# Create a retriever from the vector store
# Retrievers are the interface for querying the vector store
# Reference: https://python.langchain.com/docs/modules/data_connection/retrievers/
retriever = vector_store.as_retriever(
    search_type="similarity",  # Use cosine similarity for ranking
    search_kwargs={"k": 3}  # Retrieve top-3 most similar documents
    )
    # Other options:
    # - "mmr" (Maximal Marginal Relevance): Balances relevance with diversity
        # retriever = vector_store.as_retriever(
        # search_type="mmr",
        # search_kwargs={
        #     "k": 3,                # final number of docs returned
        #     "fetch_k": 10,         # candidates fetched before MMR re-ranks
        #     "lambda_mult": 0.5     # 0 = max diversity, 1 = max relevance
        # }
        # )
    # - "similarity_score_threshold": Only return docs above a score threshold
        # retriever = vector_store.as_retriever(
        # search_type="similarity_score_threshold",
        # search_kwargs={
        #     "score_threshold": 0.7,  # minimum similarity score (0-1)
        #     "k": 3                   # max docs to return
        # }
        # )


print("✓ Retriever configured:")
print(f"  - Search type: similarity")
print(f"  - Top-K results: 3")
print("\nTIP: k=3-5 is usually optimal. Too few → missing context, too many → noise")

# Test retriever
test_query = "Users can't log in after changing passwords"
print(f"\nTest query: '{test_query}'")
retrieved_docs = retriever.invoke(test_query)

print(f"\nRetrieved {len(retrieved_docs)} documents:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n#{i} - {doc.metadata['ticket_id']}: {doc.metadata['title']}")
    print(f"  Category: {doc.metadata['category']}")


PART 2: Setting Up Retriever
✓ Retriever configured:
  - Search type: similarity
  - Top-K results: 3

TIP: k=3-5 is usually optimal. Too few → missing context, too many → noise

Test query: 'Users can't log in after changing passwords'

Retrieved 3 documents:

#1 - TICK-001: Users unable to log in after password reset
  Category: Authentication

#2 - TICK-011: SSO authentication broken after upgrade
  Category: Authentication

#3 - TICK-016: Two-factor authentication codes not working
  Category: Authentication


In [6]:
# ============================================================================
# PART 3: Create Prompt Template with Anti-Hallucination Rules
# ============================================================================
print("\n" + "="*80)
print("PART 3: Prompt Engineering for RAG")
print("="*80)

# Define strict grounding prompt
# Prompt engineering is CRUCIAL for RAG - it tells the LLM how to use the context
# Reference: https://platform.openai.com/docs/guides/prompt-engineering
# Key principles:
# 1. Be explicit about using ONLY the provided context
# 2. Define what to do when information is missing
# 3. Request citations for transparency and verification
# 4. Set the role/persona for appropriate tone
prompt_template = """You are SupportDesk AI, a technical support assistant that helps engineers troubleshoot issues using historical support ticket data.

CRITICAL RULES:
1. Answer using ONLY information from the provided context.
2. If the question is broad or underspecified, provide the best matching known issue(s) from context and state any assumptions.
3. If context is partially relevant, still provide the most likely troubleshooting guidance from relevant tickets.
4. If the answer is truly not present in context, say "I don't have enough information in the ticket history to answer that question."
5. DO NOT make up information or use external knowledge.
6. Always cite ticket IDs for every issue/resolution you mention.
7. If multiple tickets are relevant, summarize each briefly.

Context from support tickets:
{context}

Question: {question}

Helpful Answer (with ticket citations):"""

# Convert string template to ChatPromptTemplate
# This creates a reusable template with variable placeholders
# Reference: https://python.langchain.com/docs/modules/model_io/prompts/
PROMPT = ChatPromptTemplate.from_template(prompt_template)

print("✓ Prompt template created with anti-hallucination rules:")
print("\n" + "-"*80)
print(prompt_template)
print("-"*80)


PART 3: Prompt Engineering for RAG
✓ Prompt template created with anti-hallucination rules:

--------------------------------------------------------------------------------
You are SupportDesk AI, a technical support assistant that helps engineers troubleshoot issues using historical support ticket data.

CRITICAL RULES:
1. Answer using ONLY information from the provided context.
2. If the question is broad or underspecified, provide the best matching known issue(s) from context and state any assumptions.
3. If context is partially relevant, still provide the most likely troubleshooting guidance from relevant tickets.
4. If the answer is truly not present in context, say "I don't have enough information in the ticket history to answer that question."
5. DO NOT make up information or use external knowledge.
6. Always cite ticket IDs for every issue/resolution you mention.
7. If multiple tickets are relevant, summarize each briefly.

Context from support tickets:
{context}

Question: {

In [7]:
# ============================================================================
# PART 4: Initialize LLM
# ============================================================================
print("\n" + "="*80)
print("PART 4: Initializing Language Model")
print("="*80)

# Check if OpenAI key is available
if os.getenv("OPENAI_API_KEY"):
    print("✓ OpenAI API key found")
    # Initialize ChatOpenAI for generation
    # Reference: https://python.langchain.com/docs/integrations/chat/openai
    llm = ChatOpenAI(
        model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
        temperature=0,  # Temperature controls randomness (0 = deterministic, 2 = very creative)
        # For RAG, use temperature=0 to ensure consistent, factual responses
        # Reference: https://platform.openai.com/docs/guides/text-generation/how-should-i-set-the-temperature-parameter
        timeout=120,  # Increase timeout for slower connections
        max_retries=3,  # Retry on transient failures
    )
    print(f"✓ Using {os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')}")
else:
    print("⚠ OpenAI API key not found!")
    print("  Please set OPENAI_API_KEY environment variable")
    print("  Or use Ollama: ollama pull llama2")
    print("\nFor this demo, we'll show the prompt without generating answers.")
    llm = None


PART 4: Initializing Language Model
✓ OpenAI API key found
✓ Using gpt-4o-mini


In [8]:
# ============================================================================
# PART 5: Build RAG Chain using LCEL (LangChain Expression Language)
# ============================================================================
print("\n" + "="*80)
print("PART 5: Assembling RAG Chain")
print("="*80)

# Helper function to format retrieved documents
# This concatenates all retrieved document contents into a single context string
def format_docs(docs):
    """
    Convert a list of LangChain Document objects into a single context string.

    Why this helper exists:
    - Retrievers return `List[Document]` objects.
    - Prompt templates expect plain strings for `{context}`.
    - Joining with separators keeps boundaries visible to the LLM.
    """
    # Keep document boundaries explicit so the model can attribute facts by chunk.
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

if llm:
    # Build RAG chain using LCEL (LangChain Expression Language)
    # LCEL allows you to chain components using the | operator (like Unix pipes)
    # Reference: https://python.langchain.com/docs/expression_language/
    #
    # This chain does:
    # 1. Takes a question (string input)
    # 2. Retriever gets relevant docs, format_docs combines them
    # 3. PROMPT fills in {context} and {question} variables
    # 4. LLM generates answer based on filled prompt
    # 5. StrOutputParser extracts the string response
    #
    # The dict {"context": ..., "question": ...} creates the input for the prompt
    qa_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | PROMPT
        | llm
        | StrOutputParser()
    )
    print("✓ RAG chain assembled:")
    print("  Retriever → Context Injection → LLM → Answer")
    print("\nThis is the complete RAG pipeline! Query in → Answer out")
else:
    qa_chain = None
    print("⚠ LLM not available, showing architecture only")


PART 5: Assembling RAG Chain
✓ RAG chain assembled:
  Retriever → Context Injection → LLM → Answer

This is the complete RAG pipeline! Query in → Answer out


In [19]:
# ============================================================================
# PART 7: Validation & Fallback
# ============================================================================
print("\n" + "="*80)
print("PART 7: Enhanced RAG with Answer Validation")
print("="*80)

def smart_rag(query, vector_store, qa_chain):
    """RAG with intelligent fallbacks"""
    # Get docs with scores (lower distance = better match)
    docs_with_scores = vector_store.similarity_search_with_score(query, k=3)

    if not docs_with_scores:
        return "No relevant tickets found."

    best_distance = docs_with_scores[0][1]

    if best_distance < 0.5:  # Very relevant
        ticket_id = docs_with_scores[0][0].metadata['ticket_id']
        print(f"Found possibly relevant ticket ({ticket_id}), but confidence is high.")
        return qa_chain.invoke(query)

    elif best_distance < 1.0:  # Somewhat relevant
        ticket_id = docs_with_scores[0][0].metadata['ticket_id']
        print(f"Found possibly relevant ticket ({ticket_id}), but confidence is moderate.")
        return qa_chain.invoke(query)

    else:  # Not relevant
        print("I don't have relevant ticket history for this question.")
        return qa_chain.invoke(query)




PART 7: Enhanced RAG with Answer Validation


In [20]:
print("\nTesting validation logic:")

print("\n" + "="*80)
print("\n1 How to fix database connection timeouts?")
print("\n" + "-"*80)
answer1 = smart_rag(
    "How to fix database connection timeouts?",
     vector_store,
    qa_chain
)
print("\n. Relevant query with High confidence (should answer): \n"+answer1)
print("\n" + "="*80)
print("\n2 How to fix authentication problems?")
print("\n" + "-"*80)
answer2 = smart_rag(
    "How to fix authentication problems?",
     vector_store,
    qa_chain
)
print("\n. Relevant query with Medium confidence (should answer):\n"+answer2)
print("\n" + "="*80)
print("\n3 How to fix system performance problems?")
print("\n" + "-"*80)
answer3 = smart_rag(
    "How to fix system performance problems?",
     vector_store,
    qa_chain
)
print("\n. Irrelevant query with Low confidence (should refuse): \n"+answer3)
print("\n" + "="*80)
print("\n4 What is the capital of France?")
print("\n" + "-"*80)
answer4 = smart_rag(
    "What is the capital of France?",
    vector_store,
    qa_chain
)
print("\n. Irrelevant query (should refuse): \n"+answer4)
print("\n" + "="*80)
print("\n5 how to bake cookies?")
print("\n" + "-"*80)
answer5 = smart_rag(
    "how to bake cookies?",
    vector_store,
    qa_chain
)
print("\n. Irrelevant query (should refuse): \n"+answer5)


Testing validation logic:


1 How to fix database connection timeouts?

--------------------------------------------------------------------------------
Found possibly relevant ticket (TICK-002), but confidence is high.

. Relevant query with High confidence (should answer): 
To fix database connection timeouts, you can refer to the following relevant tickets:

1. **Ticket ID: TICK-002**
   - **Issue**: The application experienced intermittent 500 errors due to 'connection pool exhausted' and 'timeout waiting for connection', affecting about 20% of requests during peak hours.
   - **Resolution**: The database connection pool was sized too small for peak load. The solution involved increasing the `max_connections` from 20 to 100, adding connection pooling monitoring and alerts, and optimizing long-running queries that were holding connections.

2. **Ticket ID: TICK-021**
   - **Issue**: Similar to TICK-002, this ticket also reported intermittent 500 errors with 'connection pool exhaust